In [3]:
2**32

4294967296

In [5]:
# introduction to classes and objects in Python

class Counter:
    def __init__(self):
        print("Counter initialized. __init__ method called.")
        self.count = 0

    def __call__(self):
        print("Counter object called. __call__ method invoked.")
        self.increment()
        return self.count

    def __str__(self):
        print("Counter object converted to string. __str__ method invoked.")
        return f"Counter value: {self.count}"
    
    def __repr__(self):
        print("Counter object representation requested. __repr__ method invoked.")
        return f"Counter(count={self.count})"

    def increment(self):
        print("Incrementing counter. increment method called.")
        self.count += 1

    def decrement(self):
        print("Decrementing counter. decrement method called.")
        self.count -= 1

    def reset(self):
        print("Resetting counter. reset method called.")
        self.count = 0

    def get_count(self):
        print("Getting counter value. get_count method called.")
        return self.count


In [6]:
c1 = Counter()
c2 = Counter()

c1()
c1()

Counter initialized. __init__ method called.
Counter initialized. __init__ method called.
Counter object called. __call__ method invoked.
Incrementing counter. increment method called.
Counter object called. __call__ method invoked.
Incrementing counter. increment method called.


2

In [7]:
print(c1) 

Counter object converted to string. __str__ method invoked.
Counter value: 2


In [8]:
c1

Counter object representation requested. __repr__ method invoked.


Counter(count=2)

In [9]:
import torch
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split

data = load_iris()
X = data.data
y = data.target

Xtrain, Xtest, ytrain, ytest = train_test_split(X, y, test_size=0.2, random_state=42)

# convert to torch tensors
Xtrain = torch.tensor(Xtrain, dtype=torch.float32) # convert to torch tensors
ytrain = torch.tensor(ytrain, dtype=torch.long) # convert to torch tensors, because pytorch needs to operate on these tensors
Xtest = torch.tensor(Xtest, dtype=torch.float32) # convert to torch tensors
ytest = torch.tensor(ytest, dtype=torch.long) # convert to torch tensors

class IrisNet(torch.nn.Module):
    '''IrisNet is a simple feedforward neural network for classifying the Iris dataset. 
    It consists of three fully connected layers with ReLU activation functions.
    This inherits from torch.nn.Module, which is the base class for all neural network modules in PyTorch.
    '''
    def __init__(self):
        super(IrisNet, self).__init__()
        self.fc1 = torch.nn.Linear(4, 10)
        self.fc2 = torch.nn.Linear(10, 5)
        self.fc3 = torch.nn.Linear(5, 3)
        self.relu = torch.nn.ReLU()

    def forward(self, x):
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        x = self.relu(x)
        x = self.fc3(x)
        return x
    
model = IrisNet()


In [10]:
criterion = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)


In [12]:
from abc import ABC
from dataclasses import dataclass

import copy
import torch
from torch import nn


# ==========================================================
# Callback Context
# ==========================================================
# Small shared object passed to callbacks instead of the full Trainer.

@dataclass
class CallbackContext:
    model: nn.Module
    epoch: int
    train_loss: float
    val_loss: float
    stop_training: bool = False
    best_state: dict | None = None


# ==========================================================
# Base Callback (no-op hooks)
# ==========================================================

class Callback(ABC):
    def on_train_begin(self, ctx):        pass
    def on_epoch_begin(self, ctx):        pass
    def on_epoch_end(self, ctx):          pass
    def on_train_end(self, ctx):          pass


# ==========================================================
# Concrete Callbacks
# ==========================================================
class Preliminary_information(Callback):

    def __init__(self, Xtrain_shape, ytrain_shape, Xval_shape, yval_shape):
        self.Xtrain_shape = Xtrain_shape
        self.ytrain_shape = ytrain_shape
        self.Xval_shape = Xval_shape
        self.yval_shape = yval_shape

    def on_train_begin(self, ctx):
        Xtrain_shape = self.Xtrain_shape
        ytrain_shape = self.ytrain_shape
        Xval_shape = self.Xval_shape
        yval_shape = self.yval_shape
        import torchinfo
        print("Training started.")
        print(f"Initial model state: {ctx.model.state_dict()}")
        print(f"Overall context: {ctx}")
        print(f"Xtrain shape: {Xtrain_shape}")
        print(f"ytrain shape: {ytrain_shape}")
        print(f"Xval shape: {Xval_shape}")
        print(f"yval shape: {yval_shape}")
        print("Model summary:")
        print(torchinfo.summary(ctx.model, input_data=torch.randn(*Xtrain_shape)))


class ProgressPrinter(Callback):

    def __init__(self, n=1):
        self.n = n

    def on_epoch_end(self, ctx):
        # only print the progress if the epoch is a multiple of n
        if ctx.epoch % self.n == 0:
            print(
                f"Epoch {ctx.epoch:4d} | "
                f"Train Loss = {ctx.train_loss:.4f} | "
                f"Val Loss = {ctx.val_loss:.4f}"
            )


class LossHistory(Callback):

    def __init__(self):
        self.train_losses = []
        self.val_losses = []

    def on_epoch_end(self, ctx):
        self.train_losses.append(ctx.train_loss)
        self.val_losses.append(ctx.val_loss)


class EarlyStopping(Callback):

    def __init__(
        self,
        patience=20,
        min_delta=0.0,
        mode="min",  # "min" for loss, "max" for metrics
        restore_best_weights=True,
    ):
        if mode not in {"min", "max"}:
            raise ValueError("mode must be 'min' or 'max'")

        self.patience = patience
        self.min_delta = min_delta
        self.mode = mode
        self.restore_best_weights = restore_best_weights

        self.best_score = float("inf") if mode == "min" else float("-inf")
        self.counter = 0

    def _improved(self, value):
        if self.mode == "min":
            return value < (self.best_score - self.min_delta)
        return value > (self.best_score + self.min_delta)

    def on_epoch_end(self, ctx):
        current = ctx.val_loss

        if self._improved(current):
            self.best_score = current
            self.counter = 0
            if self.restore_best_weights:
                ctx.best_state = copy.deepcopy(ctx.model.state_dict())
        else:
            self.counter += 1

        if self.counter >= self.patience:
            ctx.stop_training = True


# ==========================================================
# Trainer
# ==========================================================

class Trainer:

    def __init__(self, model, optimizer, criterion, callbacks=None):
        self.model = model
        self.optimizer = optimizer
        self.criterion = criterion
        self.callbacks = list(callbacks) if callbacks is not None else []

    def fit(self, Xtrain, ytrain, Xval, yval, epochs ):
        ctx = CallbackContext(model=self.model, epoch=0, train_loss=float("nan"), val_loss=float("nan"))

        for cb in self.callbacks:
            cb.on_train_begin(ctx)

        for epoch in range(epochs):
            ctx.epoch = epoch

            for cb in self.callbacks:
                cb.on_epoch_begin(ctx)

            self.model.train()
            self.optimizer.zero_grad()
            y_pred = self.model(Xtrain)
            train_loss = self.criterion(y_pred, ytrain)
            train_loss.backward()
            self.optimizer.step()
            ctx.train_loss = float(train_loss.item())

            self.model.eval()
            with torch.no_grad():
                y_val_pred = self.model(Xval)
                val_loss = self.criterion(y_val_pred, yval)
                ctx.val_loss = float(val_loss.item())

            for cb in self.callbacks:
                cb.on_epoch_end(ctx)

            if ctx.stop_training:
                print(f"Early stopping at epoch {epoch}")
                break

        if ctx.best_state is not None:
            self.model.load_state_dict(ctx.best_state)

        for cb in self.callbacks:
            cb.on_train_end(ctx)

        return ctx



history = LossHistory()
trainer = Trainer(
    model=model,
    optimizer=optimizer,
    criterion=criterion,
    callbacks=[
        ProgressPrinter(n=10),
        Preliminary_information(Xtrain_shape=Xtrain.shape, ytrain_shape=ytrain.shape, Xval_shape=Xtest.shape, yval_shape=ytest.shape),
        history,
        EarlyStopping(
            patience=20,
            min_delta=1e-4,
            mode="min",
            restore_best_weights=True,
        ),
    ],
)

ctx = trainer.fit(Xtrain, ytrain, Xtest, ytest, epochs=10000)

train_losses = history.train_losses
val_losses = history.val_losses

Training started.
Initial model state: OrderedDict({'fc1.weight': tensor([[ 0.0299, -0.2138, -0.4740,  0.3680],
        [ 0.2299,  0.2520, -0.1146, -0.4258],
        [-0.1465, -0.0804,  0.2261, -0.0978],
        [ 0.4297, -0.1585,  0.1176,  0.0776],
        [-0.1081,  0.3806, -0.4962, -0.0372],
        [ 0.0598,  0.0701, -0.3221, -0.1836],
        [ 0.4848,  0.2816, -0.3841, -0.1780],
        [ 0.0554,  0.3502,  0.4107, -0.3709],
        [-0.0448,  0.3670, -0.2229,  0.4256],
        [ 0.3387,  0.2464,  0.3808,  0.4546]]), 'fc1.bias': tensor([ 0.2294,  0.2905,  0.4549, -0.1160, -0.3701,  0.4869,  0.1234, -0.2202,
         0.2853,  0.3153]), 'fc2.weight': tensor([[-0.1600, -0.0435, -0.0098, -0.1927, -0.1542,  0.0476, -0.0315,  0.0353,
         -0.2185,  0.1994],
        [ 0.2060, -0.1328, -0.2011,  0.0914,  0.0942,  0.0455, -0.1510,  0.2236,
         -0.0623,  0.1821],
        [-0.2038, -0.2617,  0.0020,  0.1905,  0.2297,  0.0859, -0.2161,  0.2447,
          0.2233,  0.0260],
        [-0

In [19]:
ctx

CallbackContext(model=IrisNet(
  (fc1): Linear(in_features=4, out_features=10, bias=True)
  (fc2): Linear(in_features=10, out_features=5, bias=True)
  (fc3): Linear(in_features=5, out_features=3, bias=True)
  (relu): ReLU()
), epoch=28, train_loss=0.0469418503344059, val_loss=0.01797761395573616, stop_training=True, best_state=OrderedDict({'fc1.weight': tensor([[ 0.1708,  0.4229, -0.4135, -1.1872],
        [ 0.3528,  0.4434, -0.4939, -0.2583],
        [ 0.3587,  0.5276, -0.1978, -0.9963],
        [-0.4028,  0.1068, -0.2155, -0.0132],
        [ 0.7481, -0.0919,  0.6123,  0.1008],
        [ 0.4876,  0.5742, -0.7664, -0.8095],
        [-0.3497, -0.1954,  0.4250,  0.1778],
        [ 0.7074, -0.3295,  0.3479,  0.8221],
        [ 0.3155, -0.0039,  0.8283,  1.0240],
        [-0.2371, -0.0061, -0.4821,  0.0791]]), 'fc1.bias': tensor([ 2.4180,  0.9017,  2.0350, -0.4044,  1.1525,  1.0989, -0.5094, -1.2246,
        -0.5869,  0.3497]), 'fc2.weight': tensor([[ 0.9232,  0.3458,  0.7180,  0.0321,  0.